# Обучение двух моделей литотипизации керна (ДС + УФ)

ResNet18 · Adam · ReduceLROnPlateau · 8 классов

Все параметры из `config.py`. Артефакты сохраняются в `results/{RUN_NAME}/`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_recall_fscore_support, accuracy_score,
)

from config import (
    DATASET_ROOT, RESULTS_ROOT, SEED,
    BATCH_SIZE, MAX_EPOCHS, PATIENCE, MIN_DELTA, NUM_WORKERS,
    MODEL_CONFIGS, MODALITIES, CLASS_SHORT, CLASS_PALETTE,
)
from src.utils import set_seed
from src.transforms import get_transforms
from src.data import prepare_loaders
from src.models.resnet import create_resnet18
from src.training import train_one_epoch, validate, EarlyStopping, save_history, step_scheduler

# ── Имя запуска — папка внутри results/ ──────────────────────────────────────
# Меняйте для каждого нового эксперимента: 'run_01', 'run_02_dropout05', и т.д.
RUN_NAME  = 'run_01'
RUN_DIR   = RESULTS_ROOT / RUN_NAME
PLOTS_DIR = RUN_DIR / 'plots'
RUN_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CONFIGS = MODEL_CONFIGS

print(f'Device  : {DEVICE}')
print(f'Data    : {DATASET_ROOT.resolve()}')
print(f'Run     : {RUN_DIR.resolve()}')
print(f'Epochs  : {MAX_EPOCHS}  |  Patience : {PATIENCE}  |  Batch : {BATCH_SIZE}')

# Сохраняем конфиг запуска сразу
with open(RUN_DIR / 'config.json', 'w', encoding='utf-8') as f:
    json.dump({
        'run_name': RUN_NAME,
        'batch_size': BATCH_SIZE,
        'max_epochs': MAX_EPOCHS,
        'patience': PATIENCE,
        'min_delta': MIN_DELTA,
        'seed': SEED,
        'model_configs': CONFIGS,
    }, f, ensure_ascii=False, indent=2)
print('config.json сохранён')

## Обучение

In [ ]:
def train_model(modality: str, cfg: dict):
    gen = set_seed(SEED)
    print(f"\n{'='*60}")
    print(f"  Модальность : {modality}")
    print(f"  LR={cfg['lr']:.0e}  WD={cfg['wd']:.0e}  Drop={cfg['dropout']}  Freeze={cfg['freeze']}")
    print(f"  Resize={cfg['resize']}  Aug={cfg['aug']}")
    print(f"{'='*60}")

    train_loader, val_loader, classes = prepare_loaders(
        DATASET_ROOT / modality,
        train_transform=get_transforms(cfg['resize'], cfg['aug'], is_train=True),
        val_transform=get_transforms(cfg['resize'], cfg['aug'], is_train=False),
        batch_size=BATCH_SIZE, generator=gen,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'),
    )
    print(f'Классов: {len(classes)} | Трейн: {len(train_loader.dataset)} | Вал: {len(val_loader.dataset)}')

    model     = create_resnet18(len(classes), freeze_mode=cfg['freeze'], dropout_p=cfg['dropout']).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6
    )
    criterion = nn.CrossEntropyLoss()
    es        = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA)

    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, epoch, MAX_EPOCHS)
        val_mets   = validate(model, val_loader, criterion, DEVICE, epoch, MAX_EPOCHS)
        step_scheduler(scheduler, 'plateau', val_mets['f1'])

        if es.step(val_mets['f1'], epoch):
            torch.save(model.state_dict(), RUN_DIR / f'{modality}_best.pth')

        history.append({'epoch': epoch, 'train_loss': train_loss,
                         **{f'val_{k}': v for k, v in val_mets.items()}})

        cur_lr = optimizer.param_groups[0]['lr']
        print(f"Ep {epoch:3d} | TrL {train_loss:.4f} | VL {val_mets['loss']:.4f} | "
              f"F1 {val_mets['f1']:.4f} | Acc {val_mets['acc']:.4f} | lr={cur_lr:.2e} | {es.status}")

        if es.should_stop:
            print(f'\nEarly stop. Лучшая эпоха: {es.best_epoch}  F1={es.best:.4f}')
            break

    save_history(history, RUN_DIR / f'{modality}_history.json')
    torch.cuda.empty_cache()
    return history, classes


results = {}
for mod, cfg in CONFIGS.items():
    history, classes = train_model(mod, cfg)
    results[mod] = {'history': history, 'classes': classes}

## Графики обучения и метрики

Сохраняется в `results/{RUN_NAME}/plots/`

In [ ]:
# ── curves_{mod}.png — кривые обучения ───────────────────────────────────────
for mod in MODALITIES:
    hist    = results[mod]['history']
    epochs  = [h['epoch'] for h in hist]
    best_ep = max(hist, key=lambda h: h['val_f1'])

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Loss
    axes[0].plot(epochs, [h['train_loss'] for h in hist], label='Train', linewidth=2)
    axes[0].plot(epochs, [h['val_loss']   for h in hist], label='Val',   linewidth=2)
    axes[0].axvline(best_ep['epoch'], color='#94A3B8', linestyle='--', alpha=0.8)
    axes[0].set_title(f'Loss — {mod}', fontsize=12)
    axes[0].set_xlabel('Epoch'); axes[0].legend()
    for s in ['top','right']: axes[0].spines[s].set_visible(False)

    # F1
    axes[1].plot(epochs, [h['val_f1'] for h in hist], color='#22C55E', linewidth=2)
    axes[1].axvline(best_ep['epoch'], color='#94A3B8', linestyle='--', alpha=0.8,
                    label=f'best ep={best_ep["epoch"]}  F1={best_ep["val_f1"]:.3f}')
    axes[1].set_title(f'Val F1 — {mod}', fontsize=12)
    axes[1].set_xlabel('Epoch'); axes[1].legend(fontsize=9)
    axes[1].set_ylim(0, 1)
    for s in ['top','right']: axes[1].spines[s].set_visible(False)

    # Acc / Prec / Rec
    axes[2].plot(epochs, [h['val_acc']  for h in hist], label='Acc',  linewidth=2)
    axes[2].plot(epochs, [h['val_prec'] for h in hist], label='Prec', linewidth=2)
    axes[2].plot(epochs, [h['val_rec']  for h in hist], label='Rec',  linewidth=2)
    axes[2].set_title(f'Val metrics — {mod}', fontsize=12)
    axes[2].set_xlabel('Epoch'); axes[2].legend()
    axes[2].set_ylim(0, 1)
    for s in ['top','right']: axes[2].spines[s].set_visible(False)

    plt.suptitle(f'Кривые обучения — {mod}  ({RUN_NAME})', fontsize=13, y=1.02)
    plt.tight_layout()
    out = PLOTS_DIR / f'curves_{mod}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Сохранено: {out}')

In [ ]:
# ── Собираем предсказания на val для confusion + per-class F1 ─────────────────
def get_val_predictions(modality):
    cfg = CONFIGS[modality]
    classes = results[modality]['classes']
    gen = set_seed(SEED)
    _, val_loader, _ = prepare_loaders(
        DATASET_ROOT / modality,
        train_transform=get_transforms(cfg['resize'], cfg['aug'], is_train=True),
        val_transform=get_transforms(cfg['resize'], cfg['aug'], is_train=False),
        batch_size=BATCH_SIZE, generator=gen,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'),
    )
    model = create_resnet18(len(classes), freeze_mode=cfg['freeze'], dropout_p=cfg['dropout']).to(DEVICE)
    model.load_state_dict(torch.load(RUN_DIR / f'{modality}_best.pth', map_location=DEVICE, weights_only=True))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            _, preds = torch.max(model(inputs.to(DEVICE)), 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds), classes


val_preds = {mod: get_val_predictions(mod) for mod in MODALITIES}
print('Предсказания на val собраны')

In [ ]:
# ── confusion_{mod}.png — матрица ошибок на val ───────────────────────────────
for mod in MODALITIES:
    labels, preds, classes = val_preds[mod]
    short = [CLASS_SHORT.get(c, c[:8]) for c in classes]

    cm = confusion_matrix(labels, preds, normalize='true')
    fig, ax = plt.subplots(figsize=(9, 7))
    disp = ConfusionMatrixDisplay(cm, display_labels=short)
    disp.plot(ax=ax, colorbar=True, xticks_rotation=40, values_format='.2f')
    ax.set_title(f'Матрица ошибок (val, нормировано) — {mod}  [{RUN_NAME}]', fontsize=11)
    ax.set_xlabel('Предсказано', fontsize=10)
    ax.set_ylabel('Истина', fontsize=10)
    plt.tight_layout()
    out = PLOTS_DIR / f'confusion_{mod}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Сохранено: {out}')

In [ ]:
# ── per_class_f1_{mod}.png — bar chart F1 по классам ─────────────────────────
for mod in MODALITIES:
    labels, preds, classes = val_preds[mod]
    short  = [CLASS_SHORT.get(c, c[:10]) for c in classes]
    colors = [CLASS_PALETTE.get(c, '#94A3B8') for c in classes]

    f1_per = f1_score(labels, preds, labels=list(range(len(classes))),
                      average=None, zero_division=0)
    macro_f1 = f1_per.mean()

    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.bar(short, f1_per, color=colors, edgecolor='#333', linewidth=0.6)
    for bar, val in zip(bars, f1_per):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.axhline(macro_f1, color='#64748B', linestyle='--', linewidth=1.5,
               label=f'Macro avg = {macro_f1:.3f}')
    ax.set_ylim(0, 1.08)
    ax.set_title(f'F1 по классам (val) — {mod}  [{RUN_NAME}]', fontsize=12)
    ax.set_ylabel('F1 score')
    ax.tick_params(axis='x', rotation=35)
    ax.legend(fontsize=10)
    for s in ['top', 'right']:
        ax.spines[s].set_visible(False)

    plt.tight_layout()
    out = PLOTS_DIR / f'per_class_f1_{mod}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Сохранено: {out}')

In [ ]:
# ── metrics_summary.png — сводная таблица для отчёта ─────────────────────────
summary_rows = []
for mod in MODALITIES:
    hist = results[mod]['history']
    best = max(hist, key=lambda h: h['val_f1'])
    summary_rows.append([
        mod,
        f"{best['val_f1']:.4f}",
        f"{best['val_acc']:.4f}",
        f"{best['val_prec']:.4f}",
        f"{best['val_rec']:.4f}",
        f"ep {best['epoch']}",
    ])

col_labels = ['Модальность', 'F1 macro', 'Accuracy', 'Precision', 'Recall', 'Лучшая эпоха']

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.axis('off')
tbl = ax.table(
    cellText=summary_rows, colLabels=col_labels,
    cellLoc='center', loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(12)
tbl.scale(1.0, 2.0)

for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#1E3A5F')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 1:
        cell.set_facecolor('#EFF6FF')
    else:
        cell.set_facecolor('#DBEAFE')
    cell.set_edgecolor('#93C5FD')

ax.set_title(f'Итоговые метрики на val — {RUN_NAME}', fontsize=13, pad=16, fontweight='bold')
plt.tight_layout()
out = PLOTS_DIR / 'metrics_summary.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Сохранено: {out}')

# Сохраняем также JSON
summary_json = []
for mod in MODALITIES:
    best = max(results[mod]['history'], key=lambda h: h['val_f1'])
    summary_json.append({'modality': mod, **{k: v for k, v in best.items()}})
with open(RUN_DIR / 'metrics_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary_json, f, ensure_ascii=False, indent=2)
print(f'Сохранено: {RUN_DIR}/metrics_summary.json')

## Итоговая таблица в консоли

In [ ]:
print('\n' + '='*62)
print(f"{'Мод':6} {'F1':>8} {'Acc':>8} {'Prec':>8} {'Rec':>8}  Эпоха")
print('='*62)
for mod in MODALITIES:
    best = max(results[mod]['history'], key=lambda h: h['val_f1'])
    print(f"{mod:6} {best['val_f1']:8.4f} {best['val_acc']:8.4f} "
          f"{best['val_prec']:8.4f} {best['val_rec']:8.4f}  ep{best['epoch']}")
print('='*62)
print(f'\nАртефакты: {RUN_DIR}')
print(f'  config.json, {{mod}}_best.pth, {{mod}}_history.json, metrics_summary.json')
print(f'  plots/curves_{{mod}}.png, confusion_{{mod}}.png, per_class_f1_{{mod}}.png, metrics_summary.png')